# 🕵️ Dia 68 — Mapeando uma quadrilha de fraude: grafo de relações a partir de indicadores compartilhados

**Investigador Cyber 90 Dias** · Fase 5 · Rank Jutsu `A`

> Baseado no *Day 68* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-5-fraud-social-engineering/day-68.md) (*Mapping a fraud ring, a relationship graph built from shared indicators*), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Quinta, 22 de outubro. Nove cartões na parede: V1–V3 (Pix bloqueado), E1–E4 (falso emprego), BEC-B (Gráfica Jatobá) e W1 (WhatsApp clonado). Os laranjas L-01 a L-05 aparecem só como chave Pix fictícia.

Rafa: *"Cada barbante precisa de uma etiqueta com o valor exato."*

Neste notebook você vai:

1. Montar o grafo **caso-indicador** com dicionários (conjunto do original e da Operação Isca).
2. Calcular **componentes** e **centralidade de intermediação** (Brandes) só com stdlib.
3. Rodar o **teste de remoção** e achar o nó-ponte.
4. Jogar o **quiz da força do vínculo** e gerar o achado escrito + diagrama mermaid.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Todos os dados são **fictícios** (domínios `.example`, telefones `+55 00 00000-00NN`, chaves Pix `PIXFICT-*`).

## 1. Os dois conjuntos

O do original (R1–R6) e o da Operação Isca (`caso-continuo/dados/dia-68/`), embutidos para rodar offline. Mesmo formato do Gephi.

In [ ]:
NOS_ORIGINAL = 'Id,Label,Type\nR1,R1 Brightwell task scam,case\nR2,R2 Lumen task scam,case\nR3,R3 Brightwell task scam,case\nR4,R4 Cedar & Pine reshipping,case\nR5,R5 Northgate fee scam,case\nR6,R6 nanny fake check,case\nP0147,+1 202 555 0147,phone\nP0183,+1 202 555 0183,phone\nP0120,+1 312 555 0120,phone\nP0199,+1 646 555 0199,phone\nP0162,+1 415 555 0162,phone\nWA1,wallet TFICT-A1,wallet\nWB7,wallet TFICT-B7,wallet\nD1,brightwell-tasks.example,domain\nD2,lumen-rewards.example,domain\nD3,brightwel-tasks.example,domain\nD4,cedarpine-logistics.example,domain\nN1,"Brightwell Staffing",company_name\nPH1,"phrase: merchant upgrade",phrase\n'
ARESTAS_ORIGINAL = 'Source,Target,Type,Label,Weight\nR1,P0147,Undirected,texted from,3\nR1,WA1,Undirected,paid to,3\nR1,D1,Undirected,app domain,3\nR1,N1,Undirected,claimed employer,2\nR1,PH1,Undirected,used phrase,1\nR2,P0183,Undirected,texted from,3\nR2,WA1,Undirected,paid to,3\nR2,D2,Undirected,app domain,3\nR3,P0120,Undirected,texted from,3\nR3,WB7,Undirected,paid to,3\nR3,D3,Undirected,app domain,3\nR3,N1,Undirected,claimed employer,2\nR4,P0120,Undirected,recruiter phone,3\nR4,D4,Undirected,email domain,3\nR5,P0199,Undirected,recruiter phone,3\nR5,PH1,Undirected,used phrase,1\nR6,P0162,Undirected,texted from,3\n'
NOS_ISCA = 'Id,Label,Type\nV1,V1 Pix bloqueado (Serra Azul),case\nV2,V2 Pix bloqueado (Serra Azul),case\nV3,V3 Pix bloqueado (Serra Azul),case\nE1,"E1 falso emprego, recrutamento de laranja",case\nE2,E2 golpe das tarefas,case\nE3,E3 taxa de uniforme,case\nE4,E4 vaga no exterior,case\nBECB,BEC-B Grafica Jatoba imitacao,case\nW1,W1 WhatsApp clonado (Coopvale),case\nD1,serraazul-pix.example,domain\nD2,agil-tarefas.example,domain\nD3,graficajatoba-cobranca.example,domain\nT01,+55 00 00000-0001,phone\nT02,+55 00 00000-0002,phone\nT04,+55 00 00000-0004,phone\nT11,+55 00 00000-0011,phone\nT12,+55 00 00000-0012,phone\nT13,+55 00 00000-0013,phone\nT15,+55 00 00000-0015,phone\nK01,"PIXFICT-L01 (laranja L-01, chave ficticia)",pix_key\nK02,"PIXFICT-L02 (laranja L-02, chave ficticia)",pix_key\nK03,"PIXFICT-L03 (laranja L-03, chave ficticia)",pix_key\nK04,"PIXFICT-L04 (laranja L-04, chave ficticia)",pix_key\nK05,"PIXFICT-L05 (laranja L-05, chave ficticia)",pix_key\nKX9,PIXFICT-X9 (chave ficticia),pix_key\nN1,Agil Recebiveis,company_name\nPH1,frase: taxa de liberacao,phrase\n'
ARESTAS_ISCA = 'Source,Target,Type,Label,Weight\nV1,D1,Undirected,dominio da isca,3\nV1,T01,Undirected,SMS de,3\nV1,K01,Undirected,Pix para,3\nV2,D1,Undirected,dominio da isca,3\nV2,T02,Undirected,SMS de,3\nV2,K01,Undirected,Pix para,3\nV2,K02,Undirected,Pix para,3\nV3,D1,Undirected,dominio da isca,3\nV3,T04,Undirected,SMS de,3\nV3,K03,Undirected,Pix para,3\nE1,T11,Undirected,telefone do recrutador,3\nE1,K03,Undirected,conta da recrutada usada para repasse,3\nE1,N1,Undirected,empresa alegada,2\nE2,T12,Undirected,mensagem de,3\nE2,K02,Undirected,taxa paga para,3\nE2,D2,Undirected,dominio do app,3\nE2,N1,Undirected,empresa alegada,2\nE2,PH1,Undirected,usou a frase,1\nE3,T13,Undirected,telefone do recrutador,3\nE3,KX9,Undirected,taxa paga para,3\nE3,PH1,Undirected,usou a frase,1\nE4,T11,Undirected,telefone do recrutador,3\nE4,N1,Undirected,empresa alegada,2\nBECB,D3,Undirected,dominio do remetente,3\nBECB,K04,Undirected,nova chave informada,3\nW1,T15,Undirected,mensagem de,3\nW1,K05,Undirected,Pix para,3\n'

import csv, io
from collections import defaultdict

def carregar(nos_csv, arestas_csv, peso_minimo=1):
    """Devolve (rótulos, tipos, adjacência, arestas) só com arestas de peso >= peso_minimo."""
    rotulo, tipo = {}, {}
    for linha in csv.DictReader(io.StringIO(nos_csv)):
        rotulo[linha["Id"]] = linha["Label"]
        tipo[linha["Id"]] = linha["Type"]
    adj = {n: set() for n in rotulo}           # todo nó existe, mesmo sem aresta (R6, W1...)
    arestas = []
    for a in csv.DictReader(io.StringIO(arestas_csv)):
        if int(a["Weight"]) >= peso_minimo:
            adj[a["Source"]].add(a["Target"])   # não direcionado: liga nos dois sentidos
            adj[a["Target"]].add(a["Source"])
            arestas.append(a)
    return rotulo, tipo, adj, arestas

for nome, (n, a) in {"original": (NOS_ORIGINAL, ARESTAS_ORIGINAL), "Isca": (NOS_ISCA, ARESTAS_ISCA)}.items():
    rot, tip, adj, ar = carregar(n, a)
    casos = sum(1 for t in tip.values() if t == "case")
    print(f"{nome:9}: {len(rot)} nós ({casos} casos, {len(rot) - casos} indicadores), {len(ar)} arestas")

# Checkpoint do original: nenhum nó com paráfrase.
PARAFRASES = ("parecido", "similar", "mesmo grupo", "same group")
rot, _, _, _ = carregar(NOS_ISCA, ARESTAS_ISCA)
ruins = [r for r in rot.values() if any(p in r.lower() for p in PARAFRASES)]
print("Nós com paráfrase:", ruins or "nenhum ✅")

## 2. Indicadores compartilhados e agrupamentos

O achado mora no indicador que toca **dois ou mais casos**. Agrupamentos (componentes conectados) saem de uma busca em largura simples.

In [ ]:
from collections import deque

def compartilhados(tipo, adj):
    """Indicadores ligados a 2+ casos: onde está o achado."""
    return {n: sorted(v for v in adj[n] if tipo[v] == "case")
            for n in adj if tipo[n] != "case" and sum(tipo[v] == "case" for v in adj[n]) >= 2}

def agrupamentos(tipo, adj):
    """Componentes conectados, devolvendo só os casos de cada um."""
    vistos, grupos = set(), []
    for inicio in adj:
        if inicio in vistos:
            continue
        fila, comp = deque([inicio]), set()
        while fila:                             # BFS: visita tudo que é alcançável
            v = fila.popleft()
            if v in comp:
                continue
            comp.add(v)
            fila.extend(adj[v] - comp)
        vistos |= comp
        casos = sorted(c for c in comp if tipo[c] == "case")
        if casos:
            grupos.append(casos)
    return sorted(grupos, key=len, reverse=True)

PESO = {}   # (caso, indicador) -> peso, para mostrar a força de cada ligação
for a in csv.DictReader(io.StringIO(ARESTAS_ISCA)):
    PESO[(a["Source"], a["Target"])] = int(a["Weight"])
FORCA = {3: "forte", 2: "médio", 1: "fraco"}

for peso_min, titulo in ((1, "TODAS as arestas"), (3, "SÓ arestas FORTES")):
    rot, tip, adj, _ = carregar(NOS_ISCA, ARESTAS_ISCA, peso_min)
    print(f"\n=== Operação Isca — {titulo} ===")
    for ind, casos in compartilhados(tip, adj).items():
        forca = FORCA[PESO[(casos[0], ind)]]
        print(f"  {rot[ind]:45} [{forca:5}] → {', '.join(casos)}")
    print("  Agrupamentos:", agrupamentos(tip, adj))

## 3. Centralidade de intermediação (Brandes)

É o número que o Gephi calcula em *Network Diameter*. Nós com valor alto estão em muitos caminhos mais curtos entre outros nós: são candidatos a **ponte**.

In [ ]:
def intermediacao(adj):
    """Algoritmo de Brandes (2001) para grafo não direcionado e sem pesos."""
    C = dict.fromkeys(adj, 0.0)
    for s in adj:                                          # um BFS por nó de origem
        pilha, pred = [], defaultdict(list)
        sigma = dict.fromkeys(adj, 0); sigma[s] = 1        # quantos caminhos mais curtos chegam a cada nó
        dist = dict.fromkeys(adj, -1); dist[s] = 0
        fila = deque([s])
        while fila:
            v = fila.popleft(); pilha.append(v)
            for w in adj[v]:
                if dist[w] < 0:                            # primeira vez que vemos w
                    dist[w] = dist[v] + 1; fila.append(w)
                if dist[w] == dist[v] + 1:                 # v está num caminho mais curto até w
                    sigma[w] += sigma[v]; pred[w].append(v)
        delta = dict.fromkeys(adj, 0.0)
        while pilha:                                       # volta do mais longe para o mais perto
            w = pilha.pop()
            for v in pred[w]:
                delta[v] += sigma[v] / sigma[w] * (1 + delta[w])
            if w != s:
                C[w] += delta[w]
    return {n: c / 2 for n, c in C.items()}                # cada par foi contado duas vezes

def top(conjunto, peso_min=1, n=6):
    rot, tip, adj, _ = carregar(*conjunto, peso_min)
    b = intermediacao(adj)
    for no, val in sorted(b.items(), key=lambda x: -x[1])[:n]:
        print(f"   {val:7.1f}  {rot[no]}")

print("Original, todas as arestas (o original diz: R3 e P0120 perto do topo):"); top((NOS_ORIGINAL, ARESTAS_ORIGINAL))
print("\nIsca, só fortes:"); top((NOS_ISCA, ARESTAS_ISCA), 3)
print("\nIsca, todas (repare o que as arestas médias e fracas fazem com o topo):"); top((NOS_ISCA, ARESTAS_ISCA), 1)

> 💡 Com **todas** as arestas, E2 sobe para o topo da Isca: ele é o único caso que toca a frase fraca "taxa de liberação" (que puxa E3) **e** o nome médio "Agil Recebiveis". A centralidade não sabe o que é vínculo fraco. Você sabe. Por isso a leitura do achado parte das arestas **fortes**.
>
> No conjunto do original acontece o mesmo: R1 aparece acima de R3 porque a frase fraca "merchant upgrade" puxa R5 para dentro pelo R1. R3 e P0120 continuam no grupo de cima, como o original diz, e o teste de remoção abaixo mostra que são eles que seguram R4.

## 4. Teste de remoção: quem é a ponte?

Para cada nó, apague-o e veja se o maior agrupamento de casos perde alguém. Se perder, ele é ponte. (No original: apagar R3 ou P0120 tira R4 do agrupamento.)

In [ ]:
def teste_de_remocao(conjunto, peso_min):
    rot, tip, adj, _ = carregar(*conjunto, peso_min)
    base = agrupamentos(tip, adj)[0]                       # maior agrupamento intacto
    pontes = []
    for alvo in adj:
        # cópia do grafo sem o nó 'alvo'
        sem = {n: viz - {alvo} for n, viz in adj.items() if n != alvo}
        tip_sem = {n: t for n, t in tip.items() if n != alvo}
        maior = agrupamentos(tip_sem, sem)[0] if agrupamentos(tip_sem, sem) else []
        saem = [c for c in base if c != alvo and c not in maior]
        if saem:
            pontes.append((rot[alvo], saem))
    return base, pontes

for nome, conj, pmin in (("Original (todas)", (NOS_ORIGINAL, ARESTAS_ORIGINAL), 1),
                         ("Isca (só fortes)", (NOS_ISCA, ARESTAS_ISCA), 3)):
    base, pontes = teste_de_remocao(conj, pmin)
    print(f"\n{nome} — agrupamento principal: {base}")
    for rotulo, saem in pontes:
        print(f"   sem «{rotulo}» saem: {saem}")

## 5. 🎮 Quiz da força do vínculo

Dois casos compartilham o valor descrito. Responda **1** forte, **2** médio, **3** fraco ou **0** não é vínculo (nunca vira nó). `MODO_JOGO = False` mostra o gabarito.

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("A mesma carteira TFICT-A1 recebeu depósitos de R1 e R2", "1", "Uma parte controla a carteira por vez."),
    ("R1 e R3 citam a empresa 'Brightwell Staffing'", "2", "Nome de empresa: roteiros e kits são copiados."),
    ("R1 e R5 usam a frase 'merchant upgrade'", "3", "Frase curta e comum: descreve o padrão, não liga operações."),
    ("R6 e R2 foram contatados por app de mensagens", "0", "Quase todo caso tem isso: nunca vira nó."),
    ("V2 e E2 pagaram para a chave PIXFICT-L02", "1", "Chave Pix = conta: um titular por vez."),
    ("E1 e E4 receberam mensagens de +55 00 00000-0011", "1", "Telefone: controlado por uma parte por vez."),
    ("E2 e E3 falam em 'taxa de liberação'", "3", "Frase curta comum em golpe de tarefas no Brasil."),
    ("E1, E2 e E4 citam 'Agil Recebiveis'", "2", "Nome de empresa: médio sozinho."),
    ("V1 e W1 pediram pagamento por Pix", "0", "Pix é o meio de pagamento de quase todo golpe no Brasil."),
    ("O site do Dia 65 e o serraazul-pix.example estão no IP 203.0.113.77", "2",
     "Mesmo IP: forte num servidor dedicado, mas um IP pode hospedar muitos sites. Sem saber se é compartilhado, trate como médio."),
]
NOMES = {"1": "forte", "2": "médio", "3": "fraco", "0": "não é vínculo"}
acertos = 0
for descricao, certo, porque in QUIZ:
    print(f"\n• {descricao}")
    if MODO_JOGO:
        r = ""
        while r not in NOMES:
            r = input("   1 forte / 2 médio / 3 fraco / 0 não é vínculo: ").strip()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌")
    print(f"   Gabarito: {NOMES[certo]} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}  (Checkpoint BR: pelo menos 8)")

## 6. 📝 O achado escrito e o diagrama

A célula gera `notas/dia68-achado.md` com o texto no padrão do passo 5 do original e um diagrama mermaid (grosso = forte, normal = médio, pontilhado = fraco) que o GitHub desenha sozinho.

In [ ]:
#@title Achado da Operação Isca { display-mode: "form" }
confianca = "provável"  #@param ["confirmado", "provável", "contestado"]
achado = "V1, V2, V3, E1, E2 e E4 provavelmente são uma operação (fortes: serraazul-pix.example em V1–V3; chave L-01 em V1 e V2; chave L-02 em V2 e E2; chave L-03 em V3 e E1; telefone +55 00 00000-0011 em E1 e E4; médio: 'Agil Recebiveis' em E1, E2 e E4). As chaves de L-02 e L-03 são as pontes entre o golpe do Pix bloqueado e o falso emprego. E3 tem só correspondência fraca de frase. BEC-B e W1 não estão conectados."  #@param {type:"string"}

from pathlib import Path
rot, tip, adj, arestas = carregar(NOS_ISCA, ARESTAS_ISCA)
seta = {"3": "===", "2": "---", "1": "-.-"}           # espessura = força, como no original
linhas_mm = ["```mermaid", "graph LR",
             "    classDef case fill:#dbeafe,stroke:#1e40af,color:#111",
             "    classDef ind fill:#fef3c7,stroke:#92400e,color:#111"]
for no in rot:
    forma = f'(["{rot[no]}"])' if tip[no] == "case" else f'["{rot[no]}"]'
    linhas_mm.append(f"    {no}{forma}:::{'case' if tip[no] == 'case' else 'ind'}")
for a in arestas:
    linhas_mm.append(f"    {a['Source']} {seta[a['Weight']]} {a['Target']}")
linhas_mm.append("```")

texto = ("# Achado — grafo da quadrilha (Dia 68, Operação Isca, fictício)\n\n"
         f"**Confiança:** {confianca}\n\n{achado}\n\n"
         "O que este achado NÃO diz: quem é o operador; que os titulares de L-01 a L-05 são autores; "
         "nada além dos indicadores fictícios recebidos pelos canais da cooperativa.\n\n"
         + "\n".join(linhas_mm) + "\n")
Path("notas").mkdir(exist_ok=True)
saida = Path("notas/dia68-achado.md")
saida.write_text(texto, encoding="utf-8")
print(texto)
try:
    from google.colab import files
    files.download(str(saida))
except ImportError:
    print("(Fora do Colab: o arquivo ficou em notas/)")

### 🧠 Camada 8

Apofenia, viés de confirmação, ilusão de agrupamento: com barbante suficiente, tudo se liga. A tabela de força é o freio escrito antes da tentação. Escreva também o que você **não** ligou (E3) e por quê.

### Reveja a sua linha do Dia 1

Algo aqui tentou você a digitar uma chave, um telefone ou um nome em buscador ou app de banco? Anote a tentação e confirme que não agiu.

## ✅ Antes de fechar o Dia 68

- [ ] Quiz com pelo menos 8/10
- [ ] Pontes do Isca identificadas (chaves de L-02 e L-03) e o efeito da remoção explicado
- [ ] `notas/dia68-achado.md` gerado, com confiança e com o que o achado não diz
- [ ] Grafo do original montado no Gephi (R3 e P0120 no topo)

**Amanhã (Dia 69):** a chave de L-04 reaparece onde ninguém esperava: num golpe do amor que virou "investimento" em ouro.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>